# TAMAS Telemetry Analysis

Notebook para executar arquiteturas MAS sobre o benchmark **TAMAS** e analisar se sinais de telemetria capturam ataques.

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from pathlib import Path

from run_tamas_mas import run_tamas_batch_for_architecture
from datasets.tamas.tamas_features import save_tamas_feature_tables
from tamas_execution import find_tamas_file, load_jsonl, compute_features

In [ ]:
TAMAS_DATA_DIR = Path("datasets/tamas")
RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

for p in [RAW_DIR, FEATURES_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

EARLY_FRACTIONS = [0.25, 0.50, 0.75, 1.00]

RUN_EXPERIMENTS = True
REBUILD_FEATURES = True

scenario = "education"

ATTACKS = [
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]

ARCHITECTURES = [
    "centralized_tamas",
]

MODEL_NAMES = [
    "qwen3:latest",
    # "ticlazau/meta-llama-3.1-8b-instruct:latest",
]

SEEDS = [
    1, 
    2, 
    3
]

BENIGN_LIMIT = 20
ATTACK_LIMIT = 10

In [ ]:
CONDITIONS = [
    {
        "condition": "benign",
        "attack_type": "benign",
        "is_attack": False,
        "limit": BENIGN_LIMIT,
        "json_data_path": find_tamas_file("benign", scenario),
    }
]

for attack in ATTACKS:
    CONDITIONS.append(
        {
            "condition": attack,
            "attack_type": attack,
            "is_attack": True,
            "limit": ATTACK_LIMIT,
            "json_data_path": find_tamas_file(attack, scenario),
        }
    )


TODO:

1. Como computar as métricas presentes no dataset com as minhas arquiteturas

In [ ]:
all_episode_dfs = []
all_agent_dfs = []
all_tool_dfs = []
all_early_dfs = []

for llm_model in MODEL_NAMES:
    for architecture in ARCHITECTURES:
        for condition_cfg in CONDITIONS:
            
            condition = condition_cfg["condition"]
            attack_type = condition_cfg["attack_type"]
            is_attack = condition_cfg["is_attack"]
            limit = condition_cfg["limit"]
            json_data_path = condition_cfg["json_data_path"]

            for seed in SEEDS:

                print("-" * 80)
                print(
                    f"Experiment "
                    f"model={llm_model} | "
                    f"architecture={architecture} | "
                    f"condition={condition} | "
                    f"seed={seed}"
                )
                print("json_data_path:", json_data_path)

                safe_model_name = llm_model.replace(":", "_").replace("/", "_")

                raw_output_path = (
                    RAW_DIR
                    / f"tamas_{scenario}_{architecture}_{condition}_{safe_model_name}_seed_{seed}.jsonl"
                )

                raw_path = run_tamas_batch_for_architecture(
                    json_data_path=json_data_path,
                    architecture=architecture,
                    scenario=scenario,
                    limit=limit,
                    model_name=llm_model,
                    seed=[seed],
                    output_path=str(raw_output_path),
                    module_prefix="tools.tamas",
                    extra_state={
                        "forced_attack_type": attack_type,
                        "forced_is_attack": is_attack,
                    },
                )

                print("raw_path:", raw_path)

                paths = save_tamas_feature_tables(
                    raw_path=raw_path,
                    output_dir=str(FEATURES_DIR / f"{scenario}_{architecture}_{condition}_{safe_model_name}_seed_{seed}"),
                )

                print("feature paths:", paths)

                records = load_jsonl(raw_path)

                episode_df, agent_df, tool_df, early_df = compute_features(
                    records,
                    scenario,
                    attack_type,
                    seed,
                )

                for df in [episode_df, agent_df, tool_df, early_df]:
                    if not df.empty:
                        df["scenario"] = scenario
                        df["attack_type"] = attack_type
                        df["condition"] = condition
                        df["is_attack"] = is_attack
                        df["model_name"] = llm_model
                        df["architecture"] = architecture
                        df["seed"] = seed

                all_episode_dfs.append(episode_df)
                all_agent_dfs.append(agent_df)
                all_tool_dfs.append(tool_df)
                all_early_dfs.append(early_df)

In [ ]:
episode_df_all = pd.concat(all_episode_dfs, ignore_index=True) if all_episode_dfs else pd.DataFrame()
agent_df_all = pd.concat(all_agent_dfs, ignore_index=True) if all_agent_dfs else pd.DataFrame()
tool_df_all = pd.concat(all_tool_dfs, ignore_index=True) if all_tool_dfs else pd.DataFrame()
early_df_all = pd.concat(all_early_dfs, ignore_index=True) if all_early_dfs else pd.DataFrame()

episode_df_all.to_csv(FEATURES_DIR / f"episode_features_all_{safe_model_name}.csv", index=False)
agent_df_all.to_csv(FEATURES_DIR / f"agent_features_all_{safe_model_name}.csv", index=False)
tool_df_all.to_csv(FEATURES_DIR / f"tool_features_all_{safe_model_name}.csv", index=False)
early_df_all.to_csv(FEATURES_DIR / f"early_trace_features_all_{safe_model_name}.csv", index=False)

print("episode_df_all:", episode_df_all.shape)
print("agent_df_all:", agent_df_all.shape)
print("tool_df_all:", tool_df_all.shape)
print("early_df_all:", early_df_all.shape)

print("\nDistribution:")
print(episode_df_all.groupby(["condition", "is_attack"]).size())